In [ ]:
import os
import pandas as pd
import commons as c

In [ ]:
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
import os

"""
The file token.pickle stores the user's access and refresh tokens, and is
created automatically when the authorization flow completes for the first
time. If modifying these SCOPES, delete the file token.pickle.
 """
SCOPES = ['https://www.googleapis.com/auth/drive.readonly']
TOKEN_FILE = 'token.pickle'
CREDENTIALS_FILE = 'credentials.json'
PAGE_SIZE_LIMIT = 100

In [ ]:
import time
import random
from googleapiclient.errors import HttpError

def execute_with_retry(request, max_retries=5):
    for i in range(max_retries):
        try:
            return request.execute()
        except HttpError as e:
            if e.resp.status in [500, 503]:
                sleep_time = (2 ** i) + random.random()
                print(f"Retry {i+1}/{max_retries} after {sleep_time:.2f}s")
                time.sleep(sleep_time)
            else:
                raise
    raise RuntimeError("Max retries exceeded")


In [ ]:
def folder_size(service, folder_id):
    total_size = 0
    page_token = None

    while True:
        request = service.files().list(
            q=f"'{folder_id}' in parents and trashed=false",
            fields="nextPageToken, files(id,mimeType,size)",
            pageToken=page_token,
            supportsAllDrives=True,
            includeItemsFromAllDrives=True
        )

        print("Processing folder:", folder_id)
        response = execute_with_retry(request)

        for f in response.get('files', []):
            if f['mimeType'] == 'application/vnd.google-apps.folder':
                total_size += folder_size(service, f['id'])
            else:
                total_size += int(f.get('size', 0))

        page_token = response.get('nextPageToken')
        if not page_token:
            break

    return total_size


In [ ]:
if os.path.exists('token.json'):
    creds = Credentials.from_authorized_user_file('token.json', SCOPES)
else:
    flow = InstalledAppFlow.from_client_secrets_file(
        'credentials.json', SCOPES
    )
    creds = flow.run_local_server(port=0)
    with open('token.json', 'w') as token:
        token.write(creds.to_json())

In [ ]:
service = build('drive', 'v3', credentials=creds)

folders = {
    'Brisbane equiv': "1XNnFqHmF5Fv3QXNaaKfZsActRX2S3pz3",
    'Brisbane mutants': "1Nz8d3u_cf3HvRxSJ_e6PgYgmZCnjx5Td",
    'Brisbane origin': "1PON1weLj829TLMRqdgGDx0LJz8rt8FEb",
    'Brisbane extra runs': "1HCneX79jzbIFpMeg33SUw4eMSLuIX0EK",
    'Sherbrooke equiv': "1sFNMH2ky6zhMtN8xFtpwYJV7T92MSkh5",
    'Sherbrooke mutants': "10W0wuoWfVH0FOh2LOxFXQXAuv5PvtKoB",
    'Sherbrooke origin': "1a2OJ3eaJBVK3pNdneryXEYZF7nGVYiDP",
    'Sherbrooke extra runs': "1cEU1SR50KOIfo1jTNXL3PhTZzvBKNoLZ",
    'Kyiv equiv': "1DpZSMM0aj8gP7K0XQ_weGCVRw3KltHfi",
    'Kyiv mutants': "1tpX9nl0vlus-0A5_wiDPCm0ITWZIBvoe",
    'Kyiv origin': "1LtEq3rt6v2J3xSR88maOHdgreYwuE_4L",
    'Kyiv extra runs': "1IQ8uEpPg7AxmIoW_nKEfZhAOPhdI9Itd"
}

for folder_name, folder_id in folders.items():
    size_bytes = folder_size(service, folder_id)
    print(f"Size {folder_name}: {size_bytes / (1024**3):.2f} GB")

# Merge and save DFs for equiv, normal and balanced

In [ ]:
def read_and_merge_csv_files(folder_path):
    # List to hold individual DataFrames
    dataframes = []

    # Walk through all directories and subdirectories (if any)
    for root, dirs, files in os.walk(folder_path):
        for filename in files:
    # Loop through all files in the folder
    # for filename in os.listdir(folder_path):
            if filename.endswith('.csv'):
                file_path = os.path.join(root, filename)
                # file_path = os.path.join(folder_path, filename)
                # Read the CSV file into a DataFrame
                df = pd.read_csv(file_path)
                # Append the DataFrame to the list
                dataframes.append(df)

    # Concatenate all DataFrames in the list into a single DataFrame
    merged_df = pd.concat(dataframes, ignore_index=True)

    return merged_df

# Function to split the name column and create new columns
def split_name_column(name):
    name = name.replace('.qasm', '')
    parts = name.split('_')
    position = int(parts[5].replace('P', ''))
    qubit = parts[6].replace('Q', '')
    
    if len(parts) > 7: 
        parameters = parts[7].strip('[]') 
    else: 
        parameters = None

    return parts[0], parts[1], parts[3], parts[4], position, qubit, parameters

def get_gate_type(gate):
    single_qubit_gates = ["x", "h", "p", "t", "s", "z", "y", "id", "rx", "ry", "rz", "sx", "u", "u1", "u2", "u3"]
    multi_qubit_gates = ["swap", "rzz", "rxx", "cx", "cz", "cp", "ccx", "cswap", "ch"]
    if gate in single_qubit_gates:
        return 'Single_qubit'
    elif gate in multi_qubit_gates:
        return 'Multi_qubit'
    else:
        return 'Gate_not_supported'
    
# Function to categorize position based on percentage
def categorize_position(percentage):
    if percentage <= 20:
        return 'beginning'
    elif percentage <= 40:
        return 'pre_middle'
    elif percentage <= 60:
        return 'middle'
    elif percentage <= 80:
        return 'post_middle'
    else:
        return 'end'

In [ ]:
# Function to split the name column and create new columns
def split_name_column_origin(name):
    name = name.replace('.qasm', '')
    parts = name.split('_')
    return parts[0], parts[3]

In [ ]:
def get_dataframe_origin(model):
    """Generates a processed DataFrame for a given noise model, mutant type, and threshold."""

    # Get all the results in a df
    folder_path = f'./results_noise_analysis/results_{model}_origin'
    #folder_path = f'./results_{model}/results_{mutant}_{threshold}'
    df = read_and_merge_csv_files(folder_path)
    
    # Categorize input type
    df['Input_type'] = df['Input'].str.split('_').str[0]

    # Split 'Name' column into multiple columns
    df[['Algorithm', 'Qubits_number']] = df['Name'].apply(lambda x: pd.Series(split_name_column_origin(x)))

    # Drop intermediate columns
    df = df.drop(columns=['Name'])

    # Map output types
    df['Output_type'] = df['Algorithm'].map(c.output_type)

    return df

In [ ]:
def get_dataframe(model, mutant, df_char):
    """Generates a processed DataFrame for a given noise model, mutant type, and threshold."""

    # Get all the results in a df
    folder_path = f'./results_noise_analysis/results_{model}_{mutant}'
    #folder_path = f'./results_{model}/results_{mutant}_{threshold}'
    df = read_and_merge_csv_files(folder_path)
    
    # Categorize input type
    df['Input_type'] = df['Input'].str.split('_').str[0]

    # Split 'Name' column into multiple columns
    df[['Algorithm', 'Qubits_number', 'Operator', 'Gate', 'Position', 'Qubits', 'Params']] = df['Name'].apply(lambda x: pd.Series(split_name_column(x)))

    # Categorize gate type
    df['Gate_type'] = df['Gate'].apply(get_gate_type)

    # Calculate position percentage and categorize
    df['max_position'] = df.groupby(['Algorithm', 'Qubits_number'])['Position'].transform('max')
    df['position_percentage'] = (df['Position'] / df['max_position']) * 100
    df['Relative_position'] = df['position_percentage'].apply(categorize_position)

    # Drop intermediate columns
    df = df.drop(columns=['max_position', 'position_percentage', 'Name'])

    # Merge with characteristics DataFrame
    merged_df = pd.merge(df_char, df, left_on=['qubits', 'algo'], right_on=['Qubits_number', 'Algorithm'], how='right')
    merged_df = merged_df.drop(columns=['qubits', 'algo'])

    # Map output types
    merged_df['Output_type'] = merged_df['Algorithm'].map(c.output_type)

    return merged_df

In [ ]:
def process_characteristics(file_path):
    """Processes the characteristics Excel file into a DataFrame."""
    df_charac = pd.read_excel(file_path, usecols=[0, 2, 3, 5, 6, 7])
    df_charac['algo'] = df_charac.iloc[:, 0].str.split('_').str[0]
    df_charac['qubits'] = df_charac['qubits'].astype(str)
    return df_charac.drop(columns=[df_charac.columns[0]])

In [ ]:
def process_metrics(complete_df):
    """Processes metrics and saves results to CSV."""

    selected_columns = complete_df[['Algorithm', 'Qubits_number', 'hardware', 'nature']]   
    new_rows = []

    for metric, metric_name in c.metrics.items():
        metric_df = selected_columns.copy()
        metric_df['metric'] = metric
        metric_df['metric_full'] = metric_df['metric'].map(c.metric_names)
        metric_df['ideal_distance'] = complete_df[f'Ideal_{metric_name}']
        metric_df['noisy_distance'] = complete_df[f'Noisy_{metric_name}']
        metric_df['hardware_named'] = metric_df['hardware'].map(c.hardware_names)
        new_rows.append(metric_df)

    final_df = pd.concat(new_rows, ignore_index=True)
    # final_df = final_df.astype(c.type_dict)
    return final_df

In [ ]:
xlsx_path = 'data/origin_qc/programs_characteristics.xlsx'
df_charac = process_characteristics(xlsx_path)
os.makedirs('results_noise_analysis/dataframes/', exist_ok=True)
all_dfs = []

for hw in c.hardware:
    for m in ['equiv', 'normal']:
        df = get_dataframe(hw, m, df_charac)
        df['hardware'] = hw
        if m == 'equiv':
            df['nature'] = f'equivalent mutant' 
        else:
            df['nature'] = f'non-equivalent mutant' 
        all_dfs.append(df)
    
    df_origin = get_dataframe_origin(hw)
    df_origin['hardware'] = hw
    df_origin['nature'] = 'original program'  
    all_dfs.append(df_origin)

final_df = pd.concat(all_dfs, ignore_index=True)
final_df = process_metrics(final_df)
csv_path = 'results_noise_analysis/dataframes/all_data.csv'
final_df.to_csv(csv_path, mode='w', header=True, index=False)